In [0]:
# 01_clean_citibike
# Cleans Citi Bike NYC trips (October 2025). Drops rows with missing coordinates, rides shorter than 1 minute or longer than 180 minutes, and anything outside NYC. Skips JC- (Jersey City) files. Saves the result to workspace.default.citibike_clean.

In [0]:
from pyspark.sql import functions as F
base = "/Volumes/workspace/default/citibike"
df = (spark.read.option("header", True).option("inferSchema", True)
      .csv(base + "/*.csv")
      .withColumn("src", F.col("_metadata.file_path"))
      .filter(~F.col("src").contains("JC-")))
df.printSchema()
print(df.count())


root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: double (nullable = true)
 |-- start_lng: double (nullable = true)
 |-- end_lat: double (nullable = true)
 |-- end_lng: double (nullable = true)
 |-- member_casual: string (nullable = true)
 |-- src: string (nullable = false)

4731900


In [0]:
n0 = df.count()
d = df.dropna(subset=["start_lat","start_lng","end_lat","end_lng"])
n1 = d.count()
d = d.withColumn("minutes", (F.col("ended_at").cast("long") - F.col("started_at").cast("long"))/60)
d = d.filter((F.col("minutes") >= 1) & (F.col("minutes") <= 180))
n2 = d.count()
d = d.filter(F.col("start_lat").between(40.5, 40.95) & F.col("start_lng").between(-74.3, -73.65)
             & F.col("end_lat").between(40.5, 40.95) & F.col("end_lng").between(-74.3, -73.65))
d = (d.withColumn("start_hour", F.hour("started_at"))
      .withColumn("dow", F.dayofweek("started_at"))
      .drop("src"))
n3 = d.count()
print(n0, n1, n2, n3)
d.write.mode("overwrite").saveAsTable("workspace.default.citibike_clean")

4731900 4716211 4713106 4713105


In [0]:
spark.table("workspace.default.citibike_clean").count()

4713105